In [2]:
%%capture
# Load SQL extension and connect to DuckDB
%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 50
%sql duckdb://

### **Monthly Revenue Performance**

How has monthly revenue changed over time, and how many orders and customers are driving that revenue?

In [4]:
%%sql

WITH monthly_revenue AS (
SELECT 
    strftime(order_date, '%Y-%m') AS month,
    COUNT(order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(revenue), 0)::INT AS total_revenue
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY month
ORDER BY month
)

SELECT
    *
FROM monthly_revenue
WHERE month = (SELECT MIN(month) FROM monthly_revenue) OR month = (SELECT MAX(month) FROM monthly_revenue)
;

month,total_orders,unique_customers,total_revenue
2024-01,42,41,32088
2026-08,1939,1166,1466817


In the past 2.5 years, Northstar has grown from 41 customers and &#36;32,088 in revenue to 1,166 customers and &#36;1,466,817 in revenue. 


********


### Revenue by Acquisition Channel

Which customer acquisiton channels generate the most in total revenue?

In [14]:
%%sql

SELECT 
    RANK() OVER (ORDER BY SUM(o.revenue) DESC) AS rank,
    mc.channel_name AS channel_name,
    ROUND(SUM(o.revenue), 0)::INT AS total_revenue
FROM 'data/orders.csv' o
LEFT JOIN 'data/customers.csv' c
    ON o.customer_id = c.customer_id
LEFT JOIN 'data/marketing_channels.csv' mc
    ON c.channel_id = mc.channel_id
WHERE o.order_status = 'Completed'
GROUP BY channel_name
ORDER BY total_revenue DESC
;

rank,channel_name,total_revenue
1,Organic,6764476
2,Google Ads,4503094
3,Referral,4450311
4,Outbound Sales,3622330
5,Partner,1313463
6,Social,957119
7,Content,647671
8,Events,458620


Organic channels generate the most in revenue whereas Events generates the least.


--------


### Top 25 Customers by lifetime order revenue


In [13]:
%%sql

SELECT
    RANK() OVER(ORDER BY SUM(revenue) DESC) AS order_revenue_rank,
    customer_id,
    ROUND(SUM(revenue), 0)::INT AS total_order_revenue
FROM 'data/orders.csv'
WHERE order_status = 'Completed'
GROUP BY customer_id
LIMIT 25
;

order_revenue_rank,customer_id,total_order_revenue
1,11909,15970
2,14256,15079
3,10398,14604
4,19314,14113
5,13897,13421
6,19590,13020
7,17100,12905
8,11870,12708
9,13741,12437
10,19238,12158


Top 25 customers have ordered over &#36;10,000 in lifetime revenue where the top spender has almost hit &#36;16,000+.

--------------

### Repeat Customer Rate

What percentage of customers have placed more than one order? How does that percentage differ by company size?

In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
)
SELECT
    ROUND(100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS repeat_customer_pct
FROM customer_orders
;

repeat_customer_pct
85.17


In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
),
customer_sizes AS (
    SELECT
        c.company_size,
        CASE WHEN co.order_count > 1 THEN 1 ELSE 0 END AS is_repeat
    FROM 'data/customers.csv' c
    LEFT JOIN customer_orders co
        ON c.customer_id = co.customer_id
)
SELECT
    company_size,
    ROUND(100.0 * SUM(is_repeat)/COUNT(*), 2) AS repeat_customer_pct
FROM customer_sizes
GROUP BY company_size
ORDER BY repeat_customer_pct DESC
;

company_size,repeat_customer_pct
Mid-Market,82.16
Enterprise,81.7
SMB,80.59


------

### Month-over-Month Average Order Value (AOV)

How has the company's average order value changed month over month, and what is the percentage change compared to the previous month?

In [32]:
%%sql

WITH monthly_avg_revenue AS (
    SELECT 
        strftime('%Y-%m', order_date) AS month,
        AVG(revenue) AS avg_revenue,
        LAG(avg_revenue) OVER(ORDER BY month) AS previous_month_avg_revenue
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY month
)
SELECT 
    month,
    ROUND(avg_revenue, 2) AS avg_revenue,
    ROUND(previous_month_avg_revenue, 2) AS previous_month_avg_revenue,
    ROUND(avg_revenue - previous_month_avg_revenue, 2) AS avg_revenue_change,
    ROUND(100.0 * (avg_revenue - previous_month_avg_revenue) / previous_month_avg_revenue, 2) AS avg_revenue_change_pct
FROM monthly_avg_revenue
;

month,avg_revenue,previous_month_avg_revenue,avg_revenue_change,avg_revenue_change_pct
2024-01,764.01,None,None,None
2024-02,742.64,764.01,-21.36,-2.8
2024-03,663.94,742.64,-78.7,-10.6
2024-04,768.91,663.94,104.96,15.81
2024-05,754.32,768.91,-14.58,-1.9
2024-06,751.4,754.32,-2.93,-0.39
2024-07,698.46,751.4,-52.94,-7.05
2024-08,720.85,698.46,22.39,3.21
2024-09,682.15,720.85,-38.7,-5.37
2024-10,707.71,682.15,25.56,3.75


-----

### Cohort Retention Analysis

In [71]:
%%sql

WITH cohort AS (
    SELECT 
        customer_id,
        strftime('%Y-%m', MIN(order_date)) AS cohort_month
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
    GROUP BY customer_id
),
activity AS (
    SELECT DISTINCT
        customer_id,
        strftime('%Y-%m', order_date) AS activity_month
    FROM 'data/orders.csv'
    WHERE order_status = 'Completed'
),
cohort_activity AS (
    SELECT
        c.customer_id,
        c.cohort_month,
        a.activity_month,
        date_diff('month', strptime(c.cohort_month, '%Y-%m'), strptime(a.activity_month, '%Y-%m')) AS months_since_cohort
    FROM cohort c
    LEFT JOIN activity a ON c.customer_id = a.customer_id
),
cohort_retention AS (
    SELECT
        cohort_month,
        months_since_cohort,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM cohort_activity
    GROUP BY cohort_month, months_since_cohort
),
cohort_groups AS (
    SELECT 
        cohort_month,
        active_customers
    FROM cohort_retention
    WHERE months_since_cohort = 0
    ORDER BY cohort_month
)
SELECT
    cr.cohort_month,
    cr.months_since_cohort,
    cr.active_customers,
    cg.active_customers AS cohort_size,
    ROUND(100.0*cr.active_customers/cg.active_customers, 2) AS retention_rate
FROM cohort_retention cr
LEFT JOIN cohort_groups cg 
    ON cr.cohort_month = cg.cohort_month
ORDER BY cr.cohort_month, cr.months_since_cohort
;

cohort_month,months_since_cohort,active_customers,cohort_size,retention_rate
2024-01,0,41,41,100.0
2024-01,1,6,41,14.63
2024-01,2,3,41,7.32
2024-01,3,4,41,9.76
2024-01,4,4,41,9.76
2024-01,5,6,41,14.63
2024-01,6,1,41,2.44
2024-01,7,5,41,12.2
2024-01,8,1,41,2.44
2024-01,9,2,41,4.88
